Catalysts for methanol dehydrogenation

In [284]:
!pip install numpy==1.26.4 scipy==1.16.3 matplotlib scikit-learn pandas rdkit xgboost deepchem mordred pycm


## 1.3 The path to a ML model.
0. Define the task : Predict reaction rate
1. Prepare data & split data : 
2. Choose the model
3. Train the model
4. Evaluate the model
5. Use the model

In [317]:
import pandas as pd
import numpy as np

cat_data=pd.read_csv('CPEC Chemical Catalysis Database_CatTestHub_modified.csv', 
                       sep=';',               # On précise que le séparateur est le point-virgule
                       encoding='utf-8-sig')

cat_data=cat_data.dropna(axis=1, how='all')
cat_data.dropna(how='all', inplace=True)
column_feat = ['Metal', 'Support', 'Weight loading [%]','Molecular Weight of Active component [g mol-1]',  'Active Site Density [µmol gcat-1]',         
    'Dispersion [%]','Weight Hourly Space Velocity [gCH3OH gcat-1 hr-1]','Temperature [K]','Total Pressure [kPa]', 
    'Inlet Inert Used','Inert, Inlet Mole fraction [%] ', 'CO, Inlet Mole fraction [%]','CH4, Inlet Mole Fraction [%]','Methanol, Inlet Mole fraction [%]',
    'Inlet P,Methanol [kPa]', 'Inlet Inert Flow Rate [sccm]','Total Inlet Flow Rate [mol/hr]']

# Création du DataFrame features
features = cat_data[column_feat]

# Affichage
features.head()
print(cat_data.columns)


features=cat_data[column_feat]

# Cela va automatiquement transformer toutes les colonnes de type "texte" du DataFrame "features" en nouvelles colonnes de 0 et de 1.
features = pd.get_dummies(features)
features.head()

Index(['Unique Identifier', 'Metal', 'Support', 'Weight loading [%]',
       'Molecular Weight of Active component [g mol-1]', 'Catalyst ID',
       'Active Site Density [µmol gcat-1]', 'Dispersion [%]',
       'Method of determining Active Site Density', 'Pellet Diameter [µm]',
       'Weight Hourly Space Velocity [gCH3OH gcat-1 hr-1]', 'Temperature [K]',
       'Total Pressure [kPa]', 'Inlet Inert Used',
       'Inert, Inlet Mole fraction [%] ', 'CO, Inlet Mole fraction [%]',
       'CH4, Inlet Mole Fraction [%]', 'Methanol, Inlet Mole fraction [%]',
       'Inlet P,Methanol [kPa]', 'Inlet Inert Flow Rate [sccm]',
       'Total Inlet Flow Rate [mol/hr]', 'Reactor Configuration', 'Reactor ID',
       'Rate [μmol product gcat-1min-1]',
       'Confidence Interval for Rate [μmol gcat-1 min-1]',
       'STY [mol product mol active site-1 hr-1 ]', 'Selectivity CO [%]',
       'Selectivity CH4 [%]', 'Selectivity CO2 [%]', 'Conversion [%]',
       'Data Source', 'Data Submitted By', 'Fundin

,Weight loading [%],Molecular Weight of Active component [g mol-1],Active Site Density [µmol gcat-1],Dispersion [%],Weight Hourly Space Velocity [gCH3OH gcat-1 hr-1],Temperature [K],Total Pressure [kPa],"Inert, Inlet Mole fraction [%]","CO, Inlet Mole fraction [%]","CH4, Inlet Mole Fraction [%]",...,Metal_Nickel,Metal_Palladium,Metal_Platinum,Metal_Rhodium,Metal_Ruthenium,Metal_Silver,Support_Carbon,Support_Silica,Inlet Inert Used_Helium,Inlet Inert Used_Nitrogen
0,1.0,195.08,14.7,30.1,13.0,473.0,102.73,90.01,0.0,0.0,...,False,False,True,False,False,False,False,True,False,True
1,3.1,195.08,14.0,8.5,NaN,473.0,101.33,89.40,0.0,0.0,...,False,False,True,False,False,False,False,True,True,False
2,3.1,195.08,14.0,8.5,NaN,423.0,101.33,97.50,0.0,0.0,...,False,False,True,False,False,False,False,True,True,False
3,3.1,195.08,14.0,8.5,NaN,423.0,101.33,90.10,0.0,0.0,...,False,False,True,False,False,False,False,True,True,False
4,3.1,195.08,14.0,8.5,NaN,448.0,101.33,97.20,0.0,0.0,...,False,False,True,False,False,False,False,True,True,False


In [319]:
X=features
y=cat_data['Rate [μmol product gcat-1min-1]'].values

Data splitting, scaling and training of RF and XGBoost model

In [343]:
from sklearn.model_selection import train_test_split
# training data size : test data size = 0.8 : 0.2
# fixed seed using the random_state parameter, so it always has the same split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42)

In [345]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(X_train)

# save original X
X_train_ori = X_train
X_test_ori = X_test
# transform data
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [347]:
# random forest regressor, and the default criterion is mean squared error (MSE)
from sklearn.ensemble import RandomForestRegressor
ranf_reg = RandomForestRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0

# XGBoost regressor
from xgboost import XGBRegressor
xgb_reg = XGBRegressor(n_estimators=10, random_state=0)  # using 10 trees and seed=0

In [351]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

def train_test_model(model, X_train, y_train, X_test, y_test):
    """
    Function that trains a model, and tests it.
    Inputs: sklearn model, train_data, test_data
    """
    # Train model
    model.fit(X_train, y_train)
    
    # Calculate RMSE on training
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    model_train_mse = mean_squared_error(y_train, y_pred_train)
    model_test_mse = mean_squared_error(y_test, y_pred_test)
    model_train_rmse = model_train_mse ** 0.5
    model_test_rmse = model_test_mse ** 0.5
    print(f"RMSE on train set: {model_train_rmse:.3f}, and test set: {model_test_rmse:.3f}.\n")


# Train and test the random forest model
print("Evaluating Random Forest Model.")
train_test_model(ranf_reg, X_train, y_train, X_test, y_test)

# Train and test XGBoost model
print("Evaluating XGBoost model.")
train_test_model(xgb_reg, X_train, y_train, X_test, y_test)

Evaluating Random Forest Model.
RMSE on train set: 11.172, and test set: 9.271.

Evaluating XGBoost model.
RMSE on train set: 6.155, and test set: 8.637.



**k-fold method**

In [370]:
from sklearn.model_selection import KFold
n_fold=7

def train_test_kfoldmodel(model, X, y, n_fold):
    """
    Function that trains a k-fold model, and tests it.
    Inputs: sklearn model, data, n_fold
    """
    kf = KFold(n_splits=n_fold, random_state=42, shuffle=True)
    scores = []

    for i, (train_index, test_index) in enumerate(kf.split(X)):
        X_train, X_test = X.values[train_index], X.values[test_index]
        y_train, y_test = y[train_index], y[test_index]
    
        model.fit(X_train, y_train) # Entraînement du modèle
    
        predictions = model.predict(X_test) # Prédictions sur le jeu de test
    
        erreur = mean_squared_error(y_test, predictions)**0.5 # Calcul de l'erreur
        scores.append(erreur)
    
        print(f"Fold {i+1} - Erreur RMSE : {erreur:.2f}")
    print(f"Erreur moyenne sur les 5 folds : {np.mean(scores):.2f}\n")

# Train and test the random forest model
print("Evaluating Random Forest Model.")
train_test_kfoldmodel(ranf_reg, X, y, n_fold)

# Train and test XGBoost model
print("Evaluating XGBoost model.")
train_test_kfoldmodel(xgb_reg, X, y, n_fold)

Evaluating Random Forest Model.
Fold 1 - Erreur RMSE : 8.55
Fold 2 - Erreur RMSE : 6.64
Fold 3 - Erreur RMSE : 25.48
Fold 4 - Erreur RMSE : 7.22
Fold 5 - Erreur RMSE : 15.98
Fold 6 - Erreur RMSE : 26.21
Fold 7 - Erreur RMSE : 11.21
Erreur moyenne sur les 5 folds : 14.47

Evaluating XGBoost model.
Fold 1 - Erreur RMSE : 3.06
Fold 2 - Erreur RMSE : 4.15
Fold 3 - Erreur RMSE : 35.80
Fold 4 - Erreur RMSE : 8.36
Fold 5 - Erreur RMSE : 13.01
Fold 6 - Erreur RMSE : 17.02
Fold 7 - Erreur RMSE : 5.00
Erreur moyenne sur les 5 folds : 12.34



A faire : 
1. comprendre pourquoi avec k-fold c'est moins bon (why not étude etc...)
2. Essayer d'entraîner avec un y qui ne vaut pas que la rate mais l'ensemble de tout ou d'autres valeurs...